In [1]:
import sys
import os
import torch

In [2]:
# from federated_learning_cuda import load_configuration, main_experiment
from run import load_configuration, main_experiment

In [3]:
from torch_geometric.data import Data
def print_subgraph_stats(data: Data, name: str = "Subgraph"):
    """
    Prints basic statistics for a PyTorch Geometric Data object.
    
    Args:
        data: PyTorch Geometric Data object
        name: Name/identifier for the subgraph (default="Subgraph")
    """
    print(f"{name} stats:")
    print(f"Number of nodes: {data.num_nodes}")
    print(f"Number of edges: {data.edge_index.size(1)}")
    print(f"Number of features: {data.x.size(1)}")
    print(f"Number of classes: {len(torch.unique(data.y))}")
    print(f"Number of training nodes: {data.train_mask.sum().item()}")
    print(f"Number of validation nodes: {data.val_mask.sum().item()}")
    print(f"Number of test nodes: {data.test_mask.sum().item()}")
    print(f"Zero feature vectors: {(data.x.sum(dim=1) == 0).sum().item()}")
    print("-------------------")
def print_node_indices(subgraph):
    """
    Print the indices of training, validation, and test nodes in the given subgraph.

    Args:
        subgraph: A graph or subgraph object that contains train_mask, val_mask, and test_mask attributes.
    """
    print(f"Training nodes in the subgraph: {subgraph.train_mask.nonzero(as_tuple=True)[0].tolist()}")
    print(f"Number of training nodes: {subgraph.train_mask.sum().item()}")
    print(f"Validation nodes in the subgraph: {subgraph.val_mask.nonzero(as_tuple=True)[0].tolist()}")
    print(f"Number of validation nodes: {subgraph.val_mask.sum().item()}")
    print(f"Test nodes in the subgraph: {subgraph.test_mask.nonzero(as_tuple=True)[0].tolist()}")
    print(f"Number of test nodes: {subgraph.test_mask.sum().item()}")
# Example usage:
# print_subgraph_stats(subgraph, "Original subgraph")
# print_subgraph_stats(expanded_subgraph, "Expanded subgraph")

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")   
from run import load_and_split_with_khop
data, dataset, clients_data, test_data,  = load_and_split_with_khop("Cora", device, num_clients=10, beta=0.5, hop=1)
# review data for client_0
sg = clients_data[0]
print_subgraph_stats(sg, "Client 0")
print_node_indices(sg)


Client 0 stats:
Number of nodes: 344
Number of edges: 1056
Number of features: 1433
Number of classes: 5
Number of training nodes: 11
Number of validation nodes: 16
Number of test nodes: 34
Zero feature vectors: 241
-------------------
Training nodes in the subgraph: [2, 3, 5, 9, 13, 14, 15, 18, 20, 21, 22]
Number of training nodes: 11
Validation nodes in the subgraph: [23, 28, 32, 33, 34, 38, 42, 43, 54, 58, 61, 65, 71, 75, 82, 86]
Number of validation nodes: 16
Test nodes in the subgraph: [224, 226, 232, 233, 235, 244, 252, 256, 260, 262, 263, 265, 268, 269, 271, 273, 283, 285, 295, 298, 299, 303, 306, 310, 313, 316, 319, 321, 325, 333, 337, 339, 341, 343]
Number of test nodes: 34


In [5]:
# # load with fp and do the same thing
# from run import load_and_split_with_feature_prop
# data, dataset, clients_data, test_data,  = load_and_split_with_feature_prop("Cora", device, num_clients=10, beta=0.5, hop=1)
# # review data for client_0
# sg = clients_data[0]
# print_subgraph_stats(sg, "Client 0")
# print_node_indices(sg)


In [6]:
# examien a singraph from the kop thing
sg = clients_data[0]
# check number of nodes in the subgraph
print(f"Number of nodes in the subgraph: {sg.num_nodes}")
# check number of edges in the subgraph
print(f"Number of edges in the subgraph: {sg.edge_index.shape[1]}")
# check features of the subgraph
print(f"Features of the subgraph: {sg.x.shape}")
# check number of training nodes
print(f"Number of training nodes: {sg.train_mask.sum()}")
# check number of validation nodes
print(f"Number of validation nodes: {sg.val_mask.sum()}")
# check number of test nodes
print(f"Number of test nodes: {sg.test_mask.sum()}")
# number of nodes with zero feature vectors sum for each node then fidn the ones whose sum is zero
zero_features = (sg.x.sum(dim=1) == 0).sum()
print(f"Number of nodes with zero feature vectors: {zero_features}")













Number of nodes in the subgraph: 344
Number of edges in the subgraph: 1056
Features of the subgraph: torch.Size([344, 1433])
Number of training nodes: 11
Number of validation nodes: 16
Number of test nodes: 34
Number of nodes with zero feature vectors: 241


In [7]:
import torch

print(f"Number of GPUs available: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("No GPU available")

Number of GPUs available: 1
GPU 0: NVIDIA L40-48Q


In [8]:
# from federated_learning_cuda import load_configuration, main_experiment
from run import load_configuration, main_experiment
import os
import ray
ray.shutdown()

if __name__ == "__main__":
    # Define all options in a list
    # data_loading_options = ["split_dataset", "split_dataset_with_khop", "split_dataset_with_feature_prop"]
    data_loading_options = ["split_dataset_with_khop", "split_dataset_with_feature_prop", "split_dataset"]
    model_types = ["GCN"]
        
    # Load configuration once since it's common for all runs
    clients_num, beta, cfg = load_configuration()

    clients_num = 10

    # Main results directory
    main_dir = 'results'
    sub_dir = 'More_epochs_results_monte_carlo'
    # main_dir = 'New_results'
    
    dataset_name = "Cora"  # You can change this if you have different datasets

    # Loop over all combinations of data_loading_option and model_type
    for data_loading_option in data_loading_options:
        for model_type in model_types:
            # Create a structured directory for each experiment
            result_name = f"{dataset_name}_{data_loading_option}_{model_type}"
            results_dir = os.path.join(main_dir, sub_dir, result_name)

            # Create the directory if it doesn't exist
            os.makedirs(results_dir, exist_ok=True)

            print(f"Running experiment with data_loading_option: {data_loading_option}, model_type: {model_type}")
            result = main_experiment(clients_num, beta, data_loading_option, model_type, cfg, dataset_name=dataset_name, hop=2)
            
            # Replace literal '\n' with actual newline characters if necessary
            result_with_newlines = result.replace('\\n', '\n')
            
            # Create a unique filename for each combination
            filename = f'results_{dataset_name}_{data_loading_option}_{model_type}.txt'
            filepath = os.path.join(results_dir, filename)
            
            # Write the result to the text file
            with open(filepath, 'w') as f:
                f.write(result_with_newlines)
            
            print(f"Results saved to {filepath}\n")

Running experiment with data_loading_option: split_dataset_with_khop, model_type: GCN
DEVICE: cuda


2025-02-28 11:58:13,445	INFO worker.py:1816 -- Started a local Ray instance.


data_loading_option: split_dataset_with_khop
Data loaded
Device is cuda
Cora()
10


(FLClient pid=2195302) 2025-02-28 11:58:18,782 - INFO - Epoch   0| Train Loss: 1.952| Train Accuracy: 0.071
(FLClient pid=2195302) 2025-02-28 11:58:18,786 - INFO - Epoch   1| Train Loss: 1.878| Train Accuracy: 0.357
(FLClient pid=2195302) 2025-02-28 11:58:18,790 - INFO - Epoch   2| Train Loss: 1.814| Train Accuracy: 0.429
(FLClient pid=2195302) 2025-02-28 11:58:18,794 - INFO - Epoch   3| Train Loss: 1.749| Train Accuracy: 0.357
(FLClient pid=2195302) 2025-02-28 11:58:18,797 - INFO - Epoch   4| Train Loss: 1.688| Train Accuracy: 0.429
(FLClient pid=2195302) 2025-02-28 11:58:18,799 - INFO - Epoch   4| Validation Loss: 2.047, Validation Accuracy: 0.041


Training done
Round 1
Train Loss: 2.047, Train Accuracy: 0.429
Train Loss: 2.065, Train Accuracy: 0.312
Train Loss: 2.073, Train Accuracy: 0.533
Train Loss: 1.852, Train Accuracy: 0.600
Train Loss: 1.927, Train Accuracy: 0.429
Train Loss: 1.802, Train Accuracy: 0.400
Train Loss: 1.967, Train Accuracy: 0.600
Train Loss: 2.005, Train Accuracy: 0.571
Train Loss: 1.978, Train Accuracy: 0.571
Train Loss: 1.929, Train Accuracy: 0.462
Round 2
Train Loss: 2.074, Train Accuracy: 0.571
Train Loss: 2.053, Train Accuracy: 0.375
Train Loss: 2.078, Train Accuracy: 0.800
Train Loss: 1.841, Train Accuracy: 0.600
Train Loss: 1.921, Train Accuracy: 0.429
Train Loss: 1.758, Train Accuracy: 0.600
Train Loss: 1.957, Train Accuracy: 0.667
Train Loss: 1.998, Train Accuracy: 0.714
Train Loss: 1.958, Train Accuracy: 0.643
Train Loss: 1.942, Train Accuracy: 0.769
Round 3
Train Loss: 2.070, Train Accuracy: 0.643
Train Loss: 2.044, Train Accuracy: 0.375
Train Loss: 2.091, Train Accuracy: 0.800
Train Loss: 1.831, 

...

Round 1 is complete


2025-02-28 12:00:01,474	INFO worker.py:1816 -- Started a local Ray instance.


data_loading_option: split_dataset_with_feature_prop


/home/brian_bosho/FP/FP/dataprocessing/data_utils.py:59: SyntaxWarning: invalid escape sequence '\m'
  """


KeyboardInterrupt: 

### Tests